# VL-Time: toolcall

In [11]:
import os
import json
import base64
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
from openai import OpenAI
from dotenv import load_dotenv
from datetime import datetime, timedelta

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

client = OpenAI()

## Tools

### Json

In [14]:
tools = [
  {
    "type": "function",
    "function": {
      "name": "generate_consumption_series",
      "description": "Generate synthetic electricity demand (consumption) time series data.",
      "parameters": {
        "type": "object",
        "properties": {
          "start": {
            "type": "string",
            "format": "date-time",
            "description": "Start timestamp in ISO 8601 format (e.g. 2026-01-01T00:00:00)"
          },
          "end": {
            "type": "string",
            "format": "date-time",
            "description": "End timestamp in ISO 8601 format (e.g. 2026-01-01T23:00:00)"
          },
          "pattern": {
            "type": "string",
            "enum": ["residential", "industrial"],
            "description": "Consumption pattern type. Residential has peaks in the morning and evening, industrial peaks during working hours.",
            "default": "residential"
          }
        },
        "required": ["start", "end"]
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "generate_production_series",
      "description": "Generate synthetic electricity supply (production) time series data.",
      "parameters": {
        "type": "object",
        "properties": {
          "start": {
            "type": "string",
            "format": "date-time",
            "description": "Start timestamp in ISO 8601 format (e.g. 2026-01-01T00:00:00)"
          },
          "end": {
            "type": "string",
            "format": "date-time",
            "description": "End timestamp in ISO 8601 format (e.g. 2026-01-01T23:00:00)"
          },
          "pattern": {
            "type": "string",
            "enum": ["residential", "industrial"],
            "description": "Production pattern type. Residential may simulate small-scale generation (e.g. solar), industrial simulates large-scale continuous production.",
            "default": "residential"
          }
        },
        "required": ["start", "end"]
      }
    }
  }
]

### Funktionen

In [13]:
def generate_consumption_series(start, end, pattern="residential"):
    if pattern not in ["residential", "industrial"]:
        raise ValueError("pattern must be 'residential' or 'industrial'")

    start_dt = datetime.fromisoformat(start)
    end_dt = datetime.fromisoformat(end)

    if start_dt >= end_dt:
        raise ValueError("start must be before end")

    data = []
    current = start_dt

    while current < end_dt:
        hour = current.hour

        if pattern == "residential":
            base = 2 if 0 <= hour < 6 else 5 if 6 <= hour < 18 else 8
        else:  # industrial
            base = 6 if 6 <= hour < 18 else 2

        value = base + random.uniform(1, 5)

        data.append({
            "timestamp": current.isoformat(),
            "value": round(value, 2)
        })

        current += timedelta(hours=1)

    return data


def generate_production_series(start, end, pattern="residential"):
    if pattern not in ["residential", "industrial"]:
        raise ValueError("pattern must be 'residential' or 'industrial'")

    start_dt = datetime.fromisoformat(start)
    end_dt = datetime.fromisoformat(end)

    if start_dt >= end_dt:
        raise ValueError("start must be before end")

    data = []
    current = start_dt

    while current < end_dt:
        hour = current.hour

        if pattern == "residential":
            base = 1 if 0 <= hour < 6 else 4 if 6 <= hour < 18 else 2
        else:  # industrial
            base = 5 if 6 <= hour < 18 else 1

        value = base + random.uniform(0, 3)

        data.append({
            "timestamp": current.isoformat(),
            "value": round(value, 2)
        })

        current += timedelta(hours=1)

    return data

### Testaufrufe

In [15]:
start = "2024-01-01"
end = "2024-01-02"
patern = "industrial"

generate_consumption_series(start=start, end=end)
# generate_production_series(start=start, end=end)

[{'timestamp': '2024-01-01T00:00:00', 'value': 5.78},
 {'timestamp': '2024-01-01T01:00:00', 'value': 4.1},
 {'timestamp': '2024-01-01T02:00:00', 'value': 3.16},
 {'timestamp': '2024-01-01T03:00:00', 'value': 4.55},
 {'timestamp': '2024-01-01T04:00:00', 'value': 5.35},
 {'timestamp': '2024-01-01T05:00:00', 'value': 6.63},
 {'timestamp': '2024-01-01T06:00:00', 'value': 9.83},
 {'timestamp': '2024-01-01T07:00:00', 'value': 8.99},
 {'timestamp': '2024-01-01T08:00:00', 'value': 7.04},
 {'timestamp': '2024-01-01T09:00:00', 'value': 7.72},
 {'timestamp': '2024-01-01T10:00:00', 'value': 6.35},
 {'timestamp': '2024-01-01T11:00:00', 'value': 6.57},
 {'timestamp': '2024-01-01T12:00:00', 'value': 7.31},
 {'timestamp': '2024-01-01T13:00:00', 'value': 6.65},
 {'timestamp': '2024-01-01T14:00:00', 'value': 7.04},
 {'timestamp': '2024-01-01T15:00:00', 'value': 8.73},
 {'timestamp': '2024-01-01T16:00:00', 'value': 9.49},
 {'timestamp': '2024-01-01T17:00:00', 'value': 8.47},
 {'timestamp': '2024-01-01T18

## Planner

In [28]:
PLANNER_SYSTEM_PROMPT = """
You decide whether to call tools or answer directly.
"""

def run_planner(user_input, client=client, tools=tools, messages=[]):
    response = client.chat.completions.create(
        model="gpt-5.4",
        # model="gpt-5.4-mini",
        messages=[
            {"role": "system", "content": PLANNER_SYSTEM_PROMPT},
            *messages,
            {"role": "user", "content": user_input}
        ],
        tools=tools,
        tool_choice="auto"
    )

    return response

In [ ]:
user_input = "Zeige mir meinen Verbrauch von letzter Woche."
planner_response = run_planner(user_input)

# for r in planner_response:
#     print(r)

answer = planner_response.choices[0].message.content
tool_calls = planner_response.choices[0].message.tool_calls

print(answer)
print(tool_calls)

('id', 'chatcmpl-DSMco4LZ9y8g9mWk89gS86NKbcgXq')
('choices', [Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Klar — ich kann dir den Verbrauch der letzten Woche zeigen.\n\nDafür brauche ich allerdings entweder:\n1. einen konkreten Zeitraum bzw. Zeitzone, oder\n2. ich nehme standardmäßig die letzten 7 Tage bis jetzt an.\n\nWenn du möchtest, kann ich direkt den Verbrauch für die letzten 7 Tage als stündliche Zeitreihe erzeugen und dir übersichtlich zusammenfassen.  \nSag einfach: „Ja, zeig mir die letzten 7 Tage.“', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))])
('created', 1775652526)
('model', 'gpt-5.4-2026-03-05')
('object', 'chat.completion')
('service_tier', 'default')
('system_fingerprint', None)
('usage', CompletionUsage(completion_tokens=101, prompt_tokens=400, total_tokens=501, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_

In [ ]:
user_input = "Hallo"
planner_response = run_planner(user_input)

for r in planner_response:
    print(r)

('id', 'chatcmpl-DSMUQkyICjbLwyKXVTSKkFQIW3aA4')
('choices', [Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hallo! Wie kann ich dir helfen?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))])
('created', 1775652006)
('model', 'gpt-5.4-2026-03-05')
('object', 'chat.completion')
('service_tier', 'default')
('system_fingerprint', None)
('usage', CompletionUsage(completion_tokens=11, prompt_tokens=392, total_tokens=403, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))


In [41]:
answer = planner_response.choices[0].message.content
tool_calls = planner_response.choices[0].message.tool_calls

print(answer)
print(tool_calls)

Klar — ich kann dir den Verbrauch der letzten Woche zeigen.

Dafür brauche ich allerdings entweder:
1. einen konkreten Zeitraum bzw. Zeitzone, oder
2. ich nehme standardmäßig die letzten 7 Tage bis jetzt an.

Wenn du möchtest, kann ich direkt den Verbrauch für die letzten 7 Tage als stündliche Zeitreihe erzeugen und dir übersichtlich zusammenfassen.  
Sag einfach: „Ja, zeig mir die letzten 7 Tage.“
None


## Dispatcher

## Finalizer

In [8]:
FINALIZER_SYSTEM_PROMPT = """
You are a data analyst assistant.

You receive:
- user request
- tool results

Your job:
- generate a clear, structured answer
- include insights
- do NOT call tools
"""

def run_finalizer(client, user_input, tool_results):
    return client.chat.completions.create(
        model="gpt-5.3",
        messages=[
            {"role": "system", "content": FINALIZER_SYSTEM_PROMPT},
            {"role": "user", "content": user_input},
            {
                "role": "assistant",
                "content": f"Tool results:\n{tool_results}"
            }
        ]
    )

## Toolcall

In [ ]:
# toolcall

## Finalizer

In [ ]:
# finalizer

## Orchestration

In [33]:
def execute_tools(tool_calls):
    results = []

    for call in tool_calls:
        name = call["name"]
        args = call["arguments"]

        if name == "generate_consumption_series":
            result = generate_consumption_series(**args)

        elif name == "generate_production_series":
            result = generate_production_series(**args)

        else:
            raise ValueError(f"Unknown tool: {name}")

        results.append({
            "tool": name,
            "result": result
        })

    return results

def agent_loop(client, user_input, tools):
    planner_response = run_planner(client, user_input, tools)

    message = planner_response.choices[0].message

    if message.tool_calls:
        tool_calls = [
            {
                "name": tc.function.name,
                "arguments": json.loads(tc.function.arguments)
            }
            for tc in message.tool_calls
        ]

        tool_results = execute_tools(tool_calls)

        final_response = run_finalizer(client, user_input, tool_results)
        return final_response

    else:
        return planner_response